# E013 — Audio MAP relevance factor ablation

Ablation of MAP relevance factor **r ∈ {4, 8, 16, 32, 64}**.

Everything else identical to E008 +All:
- UBM 32 components, MFCC 13+Δ+ΔΔ, CMN (per-utterance)
- +All augmentation (noise SNR=20dB + speed ±10%) on train fold
- LOSO 3-fold, seed=67

**Formula:** `alpha_k = n_k / (n_k + r)` where n_k = expected target frames in component k.

With ~62 target frames/component and r=16: alpha ≈ 0.79.  
r=4 → alpha ≈ 0.94 (aggressive), r=64 → alpha ≈ 0.49 (conservative).

**E008 reference (+All, r=16):** Fold0=3.47%, Fold1=8.33%, Fold2=0.83%, Mean=4.21±3.11%, min-DCF=0.0509

In [ ]:
from pathlib import Path
import sys, copy
sys.path.insert(0, str(Path("../src").resolve()))

import numpy as np
import librosa
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_curve
from scipy.special import logsumexp
from scipy.stats import norm as scipy_norm
import pandas as pd

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

COLORS = {
    "target":    "#E74C3C",
    "nontarget": "#2E86AB",
    "green":     "#27AE60",
    "purple":    "#8E44AD",
    "gray":      "#95A5A6",
    "orange":    "#E67E22",
}

# Color palette for r values
R_VALUES = [4, 8, 16, 32, 64]
R_COLORS = {
    4:  "#E74C3C",
    8:  "#E67E22",
    16: "#27AE60",
    32: "#2E86AB",
    64: "#8E44AD",
}

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DATA = Path("../data").resolve()
manifest = load_manifest(DATA)
y_all = manifest["label"].to_numpy()
SEED = 67
print(f"{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target")

## 1. Helpers — identical to E008 +All

MFCC with CMN, +All augmentation pipeline, UBM training, MAP adaptation.  
Only `map_adapt` varies `r` — everything else is fixed.

In [ ]:
def find_wav(stem: str, data_dir: Path) -> Path:
    for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
        p = data_dir / sf / (stem + ".wav")
        if p.exists():
            return p
    raise FileNotFoundError(stem)


def aug_noise_audio(y: np.ndarray, snr_db: float = 20.0,
                    rng: np.random.Generator = None) -> np.ndarray:
    """Add white noise at target SNR (dB)."""
    signal_power = np.mean(y ** 2) + 1e-10
    noise_power  = signal_power / (10 ** (snr_db / 10))
    noise = rng.normal(0, np.sqrt(noise_power), len(y)).astype(y.dtype)
    return y + noise


def aug_speed(y: np.ndarray, rate_range=(0.9, 1.1),
              rng: np.random.Generator = None) -> np.ndarray:
    """Random time stretch without changing pitch."""
    rate = rng.uniform(*rate_range)
    return librosa.effects.time_stretch(y, rate=rate)


def extract_mfcc(y: np.ndarray, sr: int, n_mfcc: int = 13) -> np.ndarray:
    """MFCC + delta + delta-delta + CMN. Returns (T, 39)."""
    mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    mfcc   = np.vstack([mfcc, delta, delta2]).T  # (T, 39)
    mfcc  -= mfcc.mean(axis=0)                   # CMN only
    return mfcc


def extract_audio_augmented(wav_path: Path, rng: np.random.Generator):
    """Load WAV and return list of (y, sr): original + noise + speed."""
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    return [
        (y, sr),
        (aug_noise_audio(y, snr_db=20.0, rng=rng), sr),
        (aug_speed(y, rate_range=(0.9, 1.1), rng=rng), sr),
    ]


def extract_batch(df: pd.DataFrame, data_dir: Path, seed: int):
    """Extract MFCC frames with +All augmentation."""
    rng = np.random.default_rng(seed)
    all_mfcc, all_labels = [], []
    for _, row in df.iterrows():
        for y_aug, sr in extract_audio_augmented(find_wav(row["stem"], data_dir), rng):
            mfcc = extract_mfcc(y_aug, sr)
            all_mfcc.append(mfcc)
            all_labels.extend([row["label"]] * len(mfcc))
    return np.vstack(all_mfcc), np.array(all_labels)


def train_ubm(X: np.ndarray, n_components: int = 32, seed: int = 67) -> GaussianMixture:
    return GaussianMixture(
        n_components=n_components, covariance_type="diag",
        max_iter=200, random_state=seed,
    ).fit(X)


def map_adapt(ubm: GaussianMixture, X_target: np.ndarray, r: float = 16.0) -> GaussianMixture:
    """MAP adaptation of UBM means with relevance factor r."""
    log_prob  = ubm._estimate_log_prob(X_target)
    log_resp  = log_prob + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp      = np.exp(log_resp)
    n_k       = resp.sum(axis=0)                              # expected frames per component
    mu_hat    = (resp.T @ X_target) / (n_k[:, None] + 1e-10)  # target sufficient stat
    alpha     = n_k / (n_k + r)                               # adaptation coefficient
    adapted   = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted


def score_wav(wav_path: Path, adapted: GaussianMixture, ubm: GaussianMixture) -> float:
    """Score a WAV using (adapted − UBM) LLR averaged over frames."""
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    mfcc  = extract_mfcc(y, sr)
    return float((adapted.score_samples(mfcc) - ubm.score_samples(mfcc)).mean())


print("Helpers defined.")

## 2. alpha_k vs r — theoretical analysis

With ~62 target frames per component (2000 total / 32 components),
how does r map to adaptation strength?

In [ ]:
# Theoretical alpha for each r, assuming n_k = 2000/32 ≈ 62.5 frames/component
n_k_approx = 2000 / 32

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: alpha vs n_k curves for each r
ax = axes[0]
n_k_range = np.linspace(1, 500, 500)
for r in R_VALUES:
    alpha_curve = n_k_range / (n_k_range + r)
    ax.plot(n_k_range, alpha_curve, color=R_COLORS[r], lw=2, label=f"r={r}")
ax.axvline(n_k_approx, color="k", ls="--", lw=1.5, label=f"n_k≈{n_k_approx:.0f} (this dataset)")
ax.set_xlabel("Expected target frames per component (n_k)")
ax.set_ylabel("Adaptation coefficient α_k")
ax.set_title("MAP adaptation strength α_k = n_k / (n_k + r)")
ax.legend(fontsize=9)
ax.set_ylim(0, 1)

# Right: alpha at n_k=62 for each r
ax = axes[1]
alphas_at_nk = [n_k_approx / (n_k_approx + r) for r in R_VALUES]
bars = ax.bar(range(len(R_VALUES)), alphas_at_nk,
              color=[R_COLORS[r] for r in R_VALUES], alpha=0.85)
for bar, alpha_val, r in zip(bars, alphas_at_nk, R_VALUES):
    ax.text(bar.get_x() + bar.get_width()/2, alpha_val + 0.01,
            f"{alpha_val:.2f}", ha="center", fontsize=10, fontweight="bold")
ax.set_xticks(range(len(R_VALUES)))
ax.set_xticklabels([f"r={r}" for r in R_VALUES])
ax.set_ylabel("α_k at n_k ≈ 62")
ax.set_title(f"Effective adaptation α_k at n_k≈{n_k_approx:.0f} (this dataset)")
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color="k", ls=":", lw=1, label="50/50 mix")
ax.legend(fontsize=9)

plt.suptitle("E013 — MAP relevance factor r: theoretical analysis", fontsize=13)
plt.tight_layout()
plt.show()

print(f"n_k ≈ {n_k_approx:.1f} frames/component (2000 target frames / 32 components)")
print(f"{'r':>4}  {'alpha':>6}  interpretation")
print("-" * 45)
for r, alpha in zip(R_VALUES, alphas_at_nk):
    interp = "aggressive" if r <= 4 else ("moderate" if r <= 16 else "conservative")
    marker = "  ← E008 default" if r == 16 else ""
    print(f"{r:>4}  {alpha:>6.3f}  {interp}{marker}")

## 3. LOSO cross-validation — ablation over r

For each r value, run the full LOSO CV:
- Train fold: +All augmented frames (non-target → UBM, target → MAP adapt)
- Val fold: original WAVs only
- UBM: 32 components, fixed seed=67

In [ ]:
UBM_COMPONENTS = 32

all_results = {}  # r -> list of {fold, eer, min_dcf}
all_oof     = {}  # r -> oof_scores array

for r in R_VALUES:
    print(f"\n{'='*55}")
    print(f"r = {r}  (alpha ≈ {62.5/(62.5+r):.3f} at n_k=62)")
    print('='*55)

    oof_scores   = np.full(len(manifest), np.nan)
    fold_results = []

    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
        train_df = manifest.loc[train_idx]
        val_df   = manifest.loc[val_idx]

        # Extract augmented train frames (+All aug, train fold only)
        X_train, y_train = extract_batch(train_df, DATA, seed=SEED + fold_id)
        X_nt = X_train[y_train == 0]
        X_t  = X_train[y_train == 1]

        print(f"  Fold {fold_id}: {len(X_t)} target frames, {len(X_nt)} non-target frames")

        ubm     = train_ubm(X_nt, n_components=UBM_COMPONENTS, seed=SEED)
        adapted = map_adapt(ubm, X_t, r=float(r))

        # Score val on ORIGINAL WAVs only
        for idx, row in val_df.iterrows():
            oof_scores[idx] = score_wav(find_wav(row["stem"], DATA), adapted, ubm)

        val_scores = oof_scores[val_idx]
        val_labels = manifest.loc[val_idx, "label"].to_numpy()
        eer, _     = compute_eer(val_scores[val_labels == 1], val_scores[val_labels == 0])
        min_dcf, _ = compute_min_dcf(val_scores[val_labels == 1], val_scores[val_labels == 0])
        fold_results.append({"fold": fold_id, "eer": eer, "min_dcf": min_dcf})
        print(f"  → EER={eer*100:.2f}%, min-DCF={min_dcf:.4f}")

    all_results[r] = fold_results
    all_oof[r]     = oof_scores.copy()

print("\nAll r values done.")

## 4. Results — ablation table

In [ ]:
print(f"{'r':>4} {'alpha':>6} {'F0 EER':>8} {'F1 EER':>8} {'F2 EER':>8} {'Mean':>8} {'Std':>8} {'min-DCF':>9}")
print("-" * 68)

summary = []
for r in R_VALUES:
    fold_results = all_results[r]
    eers   = [fr["eer"] * 100  for fr in fold_results]
    dcfs   = [fr["min_dcf"]    for fr in fold_results]
    mean_e = np.mean(eers)
    std_e  = np.std(eers)
    mean_d = np.mean(dcfs)
    alpha  = 62.5 / (62.5 + r)
    marker = "  ← E008" if r == 16 else ""
    print(f"{r:>4} {alpha:>6.3f} {eers[0]:>8.2f} {eers[1]:>8.2f} {eers[2]:>8.2f} "
          f"{mean_e:>8.2f} {std_e:>8.2f} {mean_d:>9.4f}{marker}")
    summary.append({"r": r, "alpha": alpha, "f0": eers[0], "f1": eers[1], "f2": eers[2],
                    "mean": mean_e, "std": std_e, "min_dcf": mean_d})

print("-" * 68)

best = min(summary, key=lambda x: x["mean"])
print(f"\nBest r: {best['r']}  (alpha={best['alpha']:.3f})  "
      f"EER={best['mean']:.2f}±{best['std']:.2f}%  min-DCF={best['min_dcf']:.4f}")

# OOF overall for best r
oof_best = all_oof[best["r"]]
eer_oof, _   = compute_eer(oof_best[y_all == 1], oof_best[y_all == 0])
dcf_oof, thr = compute_min_dcf(oof_best[y_all == 1], oof_best[y_all == 0])
print(f"OOF overall (r={best['r']}): EER={eer_oof*100:.2f}%, min-DCF={dcf_oof:.4f}, threshold={thr:.3f}")

# Also print E008 reference
print(f"\nE008 reference (r=16): EER=4.21±3.11%, min-DCF=0.0509")
if best["r"] != 16:
    delta = best["mean"] - 4.21
    print(f"Best r={best['r']} vs r=16: Δ EER = {delta:+.2f}%")
    print("ACTION NEEDED: predict_audio.py and predict_fusion.py MAP_R must be updated.")
else:
    print("r=16 confirmed as optimal — no changes to predict_audio.py / predict_fusion.py needed.")

## 5. Visualizations

In [ ]:
# Main plot: mean EER ± std vs r (log-scale x-axis), mark winner and E008 baseline
means = [s["mean"] for s in summary]
stds  = [s["std"]  for s in summary]
r_arr = np.array(R_VALUES, dtype=float)

fig, ax = plt.subplots(figsize=(8, 5))

# Error band
ax.fill_between(r_arr,
                [m - s for m, s in zip(means, stds)],
                [m + s for m, s in zip(means, stds)],
                alpha=0.15, color="steelblue")

# Line
ax.plot(r_arr, means, "o-", color="steelblue", lw=2.5, ms=8, zorder=4)

# Color-coded points
for r, m, s in zip(R_VALUES, means, stds):
    ax.errorbar(r, m, yerr=s, fmt="o", color=R_COLORS[r], ms=10,
                capsize=5, elinewidth=2, zorder=5)
    ax.text(r, m + s + 0.2, f"r={r}\n{m:.2f}±{s:.2f}",
            ha="center", fontsize=8)

# Mark winner
best_r = best["r"]
best_m = best["mean"]
ax.annotate(f"★ best (r={best_r})",
            xy=(best_r, best_m),
            xytext=(best_r * 1.5, best_m + 1.5),
            arrowprops=dict(arrowstyle="->", color="goldenrod", lw=2),
            fontsize=10, color="goldenrod", fontweight="bold")

# E008 baseline (r=16)
ax.axhline(4.21, color="gray", ls="--", lw=1.5, label="E008 baseline r=16 (4.21%)")

ax.set_xscale("log")
ax.set_xticks(R_VALUES)
ax.set_xticklabels([str(r) for r in R_VALUES])
ax.set_xlabel("MAP relevance factor r (log scale)")
ax.set_ylabel("Mean EER [%] ± std")
ax.set_title("E013 — MAP r ablation: mean EER vs r")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Per-fold grouped bar chart
fig, ax = plt.subplots(figsize=(13, 5))

n_r     = len(R_VALUES)
x       = np.arange(3)   # fold 0, 1, 2
width   = 0.15

for i, r in enumerate(R_VALUES):
    fold_results = all_results[r]
    eers   = [fr["eer"] * 100 for fr in fold_results]
    offset = (i - n_r / 2 + 0.5) * width
    bars   = ax.bar(x + offset, eers, width,
                    label=f"r={r}",
                    color=R_COLORS[r], alpha=0.85)
    # Mark best r bars with gold edge
    if r == best_r:
        for bar in bars:
            bar.set_edgecolor("gold")
            bar.set_linewidth(2)

ax.set_xticks(x)
ax.set_xticklabels(["Fold 0", "Fold 1", "Fold 2"])
ax.set_ylabel("EER [%]")
ax.set_title("E013 — per-fold EER for each r value (gold border = best r)")
ax.legend(fontsize=9, loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# min-DCF mean vs r
dcf_means = [s["min_dcf"] for s in summary]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(range(len(R_VALUES)), dcf_means,
              color=[R_COLORS[r] for r in R_VALUES], alpha=0.85)
for bar, v, r in zip(bars, dcf_means, R_VALUES):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.001,
            f"{v:.4f}", ha="center", fontsize=9)
    if r == best_r:
        bar.set_edgecolor("gold")
        bar.set_linewidth(2)
ax.axhline(0.0509, color="gray", ls="--", lw=1.5, label="E008 r=16 (0.0509)")
ax.set_xticks(range(len(R_VALUES)))
ax.set_xticklabels([f"r={r}" for r in R_VALUES])
ax.set_ylabel("min-DCF (mean over folds)")
ax.set_title("E013 — min-DCF vs r")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# DET curves for all r values (OOF)
ticks     = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4]
tick_pos  = [scipy_norm.ppf(t) for t in ticks]
tick_labels = [f"{int(t*100)}" for t in ticks]

fig, ax = plt.subplots(figsize=(7, 6))

for r in R_VALUES:
    oof_s = all_oof[r]
    valid = ~np.isnan(oof_s)
    fpr, tpr, _ = roc_curve(y_all[valid], oof_s[valid])
    far_c = np.clip(fpr, 1e-4, 1 - 1e-4)
    frr_c = np.clip(1 - tpr, 1e-4, 1 - 1e-4)
    eer_c, _ = compute_eer(oof_s[valid & (y_all == 1)], oof_s[valid & (y_all == 0)])
    lw = 2.5 if r == best_r else 1.5
    ax.plot(scipy_norm.ppf(far_c), scipy_norm.ppf(frr_c),
            color=R_COLORS[r], lw=lw,
            label=f"r={r}  EER={eer_c*100:.1f}%",
            zorder=5 if r == best_r else 1)

ax.plot(tick_pos, tick_pos, "k--", lw=1)
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_labels)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_labels)
ax.set_xlabel("FAR [%]")
ax.set_ylabel("FRR [%]")
ax.set_title("E013 — DET curves, all r values (OOF)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Final summary printout for experiment log
print("=" * 65)
print("E013 — MAP r ablation: FINAL SUMMARY")
print("=" * 65)
print(f"{'r':>4} {'alpha':>6} {'F0':>7} {'F1':>7} {'F2':>7} {'Mean±std':>14} {'min-DCF':>9}")
print("-" * 65)
for s in summary:
    marker = " ★" if s["r"] == best_r else ""
    marker += "  ← E008" if s["r"] == 16 and s["r"] != best_r else (""
              if s["r"] != 16 else (" (E008=default)" if s["r"] == best_r else ""))
    print(f"{s['r']:>4} {s['alpha']:>6.3f} {s['f0']:>7.2f} {s['f1']:>7.2f} {s['f2']:>7.2f} "
          f"{s['mean']:>7.2f}±{s['std']:<5.2f} {s['min_dcf']:>9.4f}{marker}")
print("-" * 65)
print(f"\nBest r = {best_r}  (EER={best['mean']:.2f}±{best['std']:.2f}%, min-DCF={best['min_dcf']:.4f})")
print(f"E008 baseline r=16: EER=4.21±3.11%, min-DCF=0.0509")

# OOF for best r
oof_best_r = all_oof[best_r]
eer_oof_b, _    = compute_eer(oof_best_r[y_all == 1], oof_best_r[y_all == 0])
dcf_oof_b, thr_b = compute_min_dcf(oof_best_r[y_all == 1], oof_best_r[y_all == 0])
print(f"OOF overall (r={best_r}): EER={eer_oof_b*100:.2f}%, min-DCF={dcf_oof_b:.4f}, threshold={thr_b:.3f}")

if best_r != 16:
    print(f"\n⚠  Best r={best_r} ≠ 16 — update MAP_R in predict_audio.py and predict_fusion.py!")
else:
    print(f"\nr=16 confirmed as optimal — no changes needed to prediction scripts.")